In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
from IPython.display import Markdown, Image, display
P = ROOT / "data" / "processed"
F = ROOT / "docs" / "figures"
pd.options.display.float_format = "{:,.4f}".format
def table(name): return pd.read_csv(P / name)
def summary(name): return json.loads((P / name).read_text())
def fig(name): display(Image(filename=str(F / name)))
print("Simulated data. Every number below is read from executed pipeline outputs.")


Simulated data. Every number below is read from executed pipeline outputs.


# M8. Governance and monitoring (Simulated data)

These are illustrative project controls. They do not demonstrate regulatory compliance. See `docs/regulatory_notes.md`.

In [2]:
s = summary("m8_summary.json")
pd.Series({k: v for k, v in s.items()}).to_frame("value")

,value
quality_checks_passed,31
quality_checks_total,31
oot_months,9
oot_glm_auc_min,0.6896
oot_glm_auc_max,0.7084
oot_glm_actual_to_expected_min,0.8982
oot_glm_actual_to_expected_max,1.1079
psi_max,0.0995
psi_max_item,model_score
psi_red_count,0


In [3]:
table("m8_out_of_time_by_month__simulated_data.csv")

,month,model,rows,roc_auc,log_loss,brier,actual_to_expected,book
0,2024-04,glm,5108,0.6914,0.3034,0.0858,0.9541,new_business
1,2024-04,gbm,5108,0.6919,0.3030,0.0858,0.9910,new_business
2,2024-05,glm,4791,0.6989,0.2757,0.0763,0.9237,new_business
3,2024-05,gbm,4791,0.6986,0.2757,0.0762,0.9213,new_business
4,2024-06,glm,4558,0.6948,0.2606,0.0709,0.8982,new_business
5,2024-06,gbm,4558,0.6916,0.2616,0.0711,0.8937,new_business
6,2024-07,glm,4331,0.7009,0.2636,0.0718,0.9504,new_business
7,2024-07,gbm,4331,0.7045,0.2631,0.0717,0.9410,new_business
8,2024-08,glm,4545,0.6981,0.2660,0.0729,1.0343,new_business
9,2024-08,gbm,4545,0.6962,0.2662,0.0729,1.0252,new_business


In [4]:
table("m8_psi__simulated_data.csv")

,item,type,psi,rag,book
0,ln_ratio,feature,0.0408,green,new_business
1,rank,feature,0.0372,green,new_business
2,cheapest_gap,feature,0.0311,green,new_business
3,ln_comp_spread,feature,0.0007,green,new_business
4,channel,feature,0.0003,green,new_business
5,age_band,feature,0.0003,green,new_business
6,region,feature,0.0000,green,new_business
7,risk_band,feature,0.0000,green,new_business
8,model_score,score,0.0995,green,new_business
9,ln_change,feature,0.0007,green,renewal


In [5]:
table("m8_stress_tests__simulated_data.csv")

,scenario,baseline_contribution,optimised_contribution,gain,baseline_margin,nb_policies_vs_baseline,renewal_policies_vs_baseline,optimised_margin,baseline_margin_floor_met,margin_floor_met,nb_volume_floor_met,max_nb_rank_deterioration,max_renewal_change
0,central,"1,482,712.9845","1,559,466.9599",0.0518,0.2314,0.9602,0.9812,0.2440,True,True,True,0.3992,0.0999
1,elasticity x0.7,"1,482,712.9845","1,565,842.7832",0.0561,0.2314,0.9688,0.9869,0.2438,True,True,True,0.3992,0.0999
2,elasticity x1.3,"1,482,712.9845","1,553,632.8781",0.0478,0.2314,0.9547,0.9753,0.2441,True,True,True,0.3992,0.0999
3,competitor prices -5%,"1,418,576.2170","1,488,923.2660",0.0496,0.2344,0.9624,0.9800,0.2469,True,True,True,0.4554,0.0999
4,competitor prices +5%,"1,550,592.8592","1,635,540.0772",0.0548,0.2276,0.9575,0.9828,0.2403,True,True,True,0.4308,0.0999
5,cost per policy +5%,"1,242,430.8363","1,323,493.8536",0.0652,0.1939,0.9602,0.9812,0.2071,False,False,True,0.3992,0.0999
6,cost per policy +10%,"1,002,148.6880","1,087,520.7472",0.0852,0.1564,0.9602,0.9812,0.1701,False,False,True,0.3992,0.0999
7,"combined: elasticity x1.3, competitors -5%, co...","964,848.8906","1,039,256.2390",0.0771,0.1594,0.9560,0.9738,0.1732,False,False,True,0.4554,0.0999


Three stress cases start with a baseline margin already below the floor, so the floor flag there is not caused by the decision. The summary separates the two counts.

In [6]:
table("m8_model_risk_register__simulated_data.csv")

,id,area,risk,rating,impact,control,status
0,R01,Data,All customer-level data is simulated from illu...,High,"Results show method, not market truth. Any rea...",Label every output 'Simulated data'. Calibrati...,Open by design
1,R02,Elasticity,Observational price effects are biased when pr...,High,Observational GLM and GBM elasticities are les...,Randomised price test. Optimiser uses the expe...,Mitigated in simulation. Needs a live test in ...
2,R03,Elasticity,Randomised test is too thin to resolve small c...,Medium,Broker arm confidence interval spans zero. Cal...,Pool thin channels. Report intervals. Increase...,Open
3,R04,Retention,"No randomised test for renewal price changes, ...",High,Renewal response may be biased. Renewal moves ...,Cap renewal moves. Stress elasticity. Add a re...,Open
4,R05,Profitability,Cost per policy is a fixed configuration value...,High,Contribution is a proxy. Margin conclusions de...,Excluded by design: no claims model. Cost shif...,Accepted scope limit
5,R06,Optimisation,SLSQP finds a local optimum on a piecewise smo...,Medium,Another start could give a different solution.,"Smoothed rank, analytic Jacobians checked agai...",Mitigated
6,R07,Optimisation,Rank is smoothed inside the optimiser but exac...,Low,Small gap between solver and evaluated constra...,Evaluate on exact rank. Report slack after exa...,Mitigated
7,R08,Model,Out-of-time drift in competitor position or mix,Medium,Conversion model degrades as the market moves.,Monthly out-of-time metrics and PSI with amber...,Monitored
8,R09,Model,Gradient boosting can extrapolate poorly outsi...,Medium,Price-response curves from GBM may be unreliab...,Monotone constraints on price features. GLM us...,Mitigated
9,R10,Market data,Competitor view limited to eight simulated com...,Medium,Real PCW lists vary in length and include insu...,Manual quote route for sense checks only. Neve...,Open
